# Job Salary Estimator

## DAY 1: Data Curation

Build a model that predicts job salary from job descriptions, using the lukebarousse/data_jobs dataset.

**Dataset**: https://huggingface.co/datasets/lukebarousse/data_jobs

Today we'll load, scrub, curate, and sample our job salary data.

**Business value of Data Curation**  
Data curation is where the science happens. R&D with your dataset often has greater impact on performance than hyperparameter optimization. Quality time with data quality!

In [ ]:
# imports
import os
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np
import random
from job_salary.items import Job
from job_salary.parser import parse

load_dotenv(override=True)

In [ ]:
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    login(hf_token, add_to_git_credential=True)

In [ ]:
# Load the data_jobs dataset (streaming for large size ~785k rows)
dataset = load_dataset("lukebarousse/data_jobs", split="train", trust_remote_code=True)

In [ ]:
print(f"Total rows: {len(dataset):,}")

In [ ]:
dataset[0]

In [ ]:
# Parse into Job objects - filter by salary range, min text length
items = []
for dp in tqdm(dataset, desc="Parsing"):
    job = parse(dp)
    if job is not None:
        items.append(job)

print(f"Parsed {len(items):,} jobs from {len(dataset):,} rows")

In [ ]:
items[0]

In [ ]:
salaries = [j.salary for j in items]
lengths = [len(j.full) for j in items]

plt.figure(figsize=(15, 6))
plt.title(f"Salary distribution: Avg ${sum(salaries)/len(salaries):,.0f}, Max ${max(salaries):,.0f}")
plt.xlabel('Salary ($)')
plt.ylabel('Count')
plt.hist(salaries, rwidth=0.7, color='steelblue', bins=50)
plt.show()

In [ ]:
plt.figure(figsize=(15, 6))
plt.title(f"Text length: Avg {sum(lengths)/len(lengths):,.0f}, Max {max(lengths):,}")
plt.xlabel('Length (chars)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color='coral', bins=50)
plt.show()

In [ ]:
# Deduplicate by title and full text
random.seed(42)
random.shuffle(items)

seen = set()
items = [x for x in tqdm(items) if not (x.title in seen or seen.add(x.title))]

seen = set()
items = [x for x in tqdm(items) if not (x.full in seen or seen.add(x.full))]

del seen
print(f"After deduplication: {len(items):,} jobs")

In [ ]:
from collections import Counter
cat_counts = Counter(j.category for j in items)
plt.figure(figsize=(12, 5))
cats, counts = zip(*cat_counts.most_common(15))
plt.bar(cats, counts, color='teal')
plt.xticks(rotation=45, ha='right')
plt.title('Jobs by category')
plt.show()

In [ ]:
# Stratified sample - bias toward higher salaries (rarer)
SIZE = min(100_000, len(items))  # cap for speed

salaries_arr = np.array([j.salary for j in items], dtype=float)
categories_arr = np.array([j.category for j in items])
p = (salaries_arr - salaries_arr.min()) / (salaries_arr.max() - salaries_arr.min() + 1e-9)
w = p ** 2  # favor higher salaries
w = w / w.sum()
idx = np.random.choice(len(items), size=SIZE, replace=False, p=w)
sample = [items[i] for i in idx]

random.seed(42)
random.shuffle(sample)
print(f"Sample size: {len(sample):,}")

In [ ]:
# Train / val / test split
n = len(sample)
train = sample[: int(0.8 * n)]
val = sample[int(0.8 * n) : int(0.9 * n)]
test = sample[int(0.9 * n) :]

print(f"Train: {len(train):,}, Val: {len(val):,}, Test: {len(test):,}")

In [ ]:
# Optional: Push to HuggingFace Hub (replace with your username)
# username = "your_hf_username"
# Job.push_to_hub(f"{username}/job_salaries", train, val, test)